# Pairwise Wasserstein Distances after Burn-In

In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Configuration
FOLDERS = ["paths1", "paths2"]
OUTDIR = Path("AVMs6")
OUTDIR.mkdir(exist_ok=True)

BURN_IN = 50_000
QUANTILE_GRID_SIZE = 4001
PAIRWISE_BLOCK_SIZE = 8
QUANTILE_DTYPE = np.float32
OBSERVABLES = ["theta1", "w1", "theta2", "w2"]

# Common u-grid and trapezoid weights for the quantile-integral approximation of W1.
U_GRID = np.linspace(0.0, 1.0, QUANTILE_GRID_SIZE, dtype=np.float64)
TRAPZ_WEIGHTS = np.ones(QUANTILE_GRID_SIZE, dtype=np.float32)
TRAPZ_WEIGHTS[0] = 0.5
TRAPZ_WEIGHTS[-1] = 0.5
DELTA_U = float(U_GRID[1] - U_GRID[0])


def read_csv_4cols(file_path: str) -> np.ndarray:
    """Read the first four observable columns as float64."""
    df = pd.read_csv(file_path, usecols=[0, 1, 2, 3])
    arr = df.to_numpy(dtype=np.float64, copy=False)
    if arr.shape[1] != 4:
        raise ValueError(f"Expected 4 observable columns in {file_path}, got {arr.shape[1]}")
    return arr


def load_burned_trajectories(folder: str, burn_in: int) -> tuple[list[str], list[np.ndarray]]:
    """Load all trajectories in a folder and drop the burn-in prefix."""
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {folder}/")

    file_names: list[str] = []
    trajectories: list[np.ndarray] = []

    for idx, fp in enumerate(files, start=1):
        if idx % 100 == 0 or idx == len(files):
            print(f"[{folder}] loaded {idx}/{len(files)} files")

        arr = read_csv_4cols(fp)
        if arr.shape[0] <= burn_in:
            raise ValueError(
                f"Burn-in {burn_in} leaves no data in {fp} (length={arr.shape[0]})"
            )

        burned = arr[burn_in:, :]
        file_names.append(os.path.basename(fp))
        trajectories.append(burned)

    return file_names, trajectories


def linear_quantiles_1d(x: np.ndarray, u_grid: np.ndarray) -> np.ndarray:
    """Empirical quantiles on a fixed grid using linear interpolation on the sorted sample."""
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        raise ValueError("Quantiles are undefined for an empty or non-finite sample.")

    xs = np.sort(x)
    if xs.size == 1:
        return np.full(u_grid.shape, xs[0], dtype=np.float64)

    positions = u_grid * (xs.size - 1)
    lower = np.floor(positions).astype(np.int64)
    upper = np.ceil(positions).astype(np.int64)
    weight = positions - lower

    return xs[lower] * (1.0 - weight) + xs[upper] * weight


def build_quantile_matrix(
    trajectories: list[np.ndarray],
    observable_idx: int,
    u_grid: np.ndarray,
    dtype: np.dtype = np.float32,
) -> np.ndarray:
    """Precompute the empirical quantile curve for one observable in every trajectory."""
    n_paths = len(trajectories)
    qmat = np.empty((n_paths, u_grid.size), dtype=dtype)

    for idx, arr in enumerate(trajectories, start=1):
        if idx % 100 == 0 or idx == n_paths:
            print(f"    quantiles {idx}/{n_paths}")

        qmat[idx - 1, :] = linear_quantiles_1d(arr[:, observable_idx], u_grid).astype(dtype, copy=False)

    return qmat


def upper_triangular_w1_matrix_from_quantiles(
    quantiles: np.ndarray,
    delta_u: float,
    trapz_weights: np.ndarray,
    block_size: int,
) -> np.ndarray:
    """Compute an upper-triangular pairwise W1 matrix from quantile curves."""
    n_paths = quantiles.shape[0]
    out = np.full((n_paths, n_paths), np.nan, dtype=np.float32)
    np.fill_diagonal(out, 0.0)

    for row_start in range(0, n_paths, block_size):
        row_stop = min(row_start + block_size, n_paths)
        row_block = quantiles[row_start:row_stop]
        print(f"    pairwise rows {row_start + 1}-{row_stop} / {n_paths}")

        for col_start in range(row_start, n_paths, block_size):
            col_stop = min(col_start + block_size, n_paths)
            col_block = quantiles[col_start:col_stop]

            diff = np.abs(row_block[:, None, :] - col_block[None, :, :])
            block = delta_u * np.einsum("ijk,k->ij", diff, trapz_weights, optimize=True)

            if row_start == col_start:
                masked_block = np.full(block.shape, np.nan, dtype=np.float32)
                tri = np.triu_indices(block.shape[0], k=1)
                masked_block[tri] = block[tri].astype(np.float32, copy=False)
                np.fill_diagonal(masked_block, 0.0)
                out[row_start:row_stop, col_start:col_stop] = masked_block
            else:
                out[row_start:row_stop, col_start:col_stop] = block.astype(np.float32, copy=False)

    return out


def write_result_workbook(output_path: Path, matrices: dict[str, pd.DataFrame]) -> None:
    """Write one Excel workbook with one sheet per observable."""
    with pd.ExcelWriter(output_path) as writer:
        for sheet_name, df in matrices.items():
            df.to_excel(writer, sheet_name=sheet_name)


def process_folder(folder: str) -> Path:
    print(f"\n=== {folder}: loading trajectories with burn-in={BURN_IN:,} ===")
    file_names, trajectories = load_burned_trajectories(folder, burn_in=BURN_IN)
    output_path = OUTDIR / f"{folder}_results.xlsx"

    workbook_sheets: dict[str, pd.DataFrame] = {}

    for observable_idx, observable_name in enumerate(OBSERVABLES):
        print(f"\n[{folder}] Observable: {observable_name}")
        quantiles = build_quantile_matrix(
            trajectories=trajectories,
            observable_idx=observable_idx,
            u_grid=U_GRID,
            dtype=QUANTILE_DTYPE,
        )

        matrix = upper_triangular_w1_matrix_from_quantiles(
            quantiles=quantiles,
            delta_u=DELTA_U,
            trapz_weights=TRAPZ_WEIGHTS,
            block_size=PAIRWISE_BLOCK_SIZE,
        )

        workbook_sheets[observable_name] = pd.DataFrame(
            matrix,
            index=file_names,
            columns=file_names,
        )

    print(f"\n[{folder}] writing workbook: {output_path}")
    write_result_workbook(output_path, workbook_sheets)
    print(f"[{folder}] done")
    return output_path



In [ ]:
result_paths = [process_folder(folder) for folder in FOLDERS]
result_paths

In [2]:
from scipy.stats import wasserstein_distance

# Same inputs and workbook structure as above, but this uses SciPy's exact 1D Wasserstein distance.
# The `_scipy` suffix avoids overwriting the trapezoid-approximation outputs.
# SCIPY_OUTPUT_SUFFIX = "_scipy"


def upper_triangular_w1_matrix_scipy(
    trajectories: list[np.ndarray],
    observable_idx: int,
) -> np.ndarray:
    """Compute the upper-triangular pairwise W1 matrix with SciPy."""
    n_paths = len(trajectories)
    out = np.full((n_paths, n_paths), np.nan, dtype=np.float32)
    np.fill_diagonal(out, 0.0)

    series = []
    for arr in trajectories:
        x = np.asarray(arr[:, observable_idx], dtype=np.float64)
        x = x[np.isfinite(x)]
        if x.size == 0:
            raise ValueError(f"Observable {observable_idx} contains no finite values in one trajectory.")
        series.append(x)

    for i in range(n_paths):
        if (i + 1) % 10 == 0 or i + 1 == n_paths:
            print(f"    pairwise rows {i + 1}/{n_paths}")

        xi = series[i]
        for j in range(i + 1, n_paths):
            out[i, j] = float(wasserstein_distance(xi, series[j]))

    return out


def process_folder_scipy(folder: str) -> Path:
    print(f"\n=== {folder}: SciPy pairwise W1 with burn-in={BURN_IN:,} ===")
    file_names, trajectories = load_burned_trajectories(folder, burn_in=BURN_IN)
    output_path = OUTDIR / f"{folder}_results.xlsx"

    workbook_sheets: dict[str, pd.DataFrame] = {}

    for observable_idx, observable_name in enumerate(OBSERVABLES):
        print(f"\n[{folder}] Observable: {observable_name}")
        matrix = upper_triangular_w1_matrix_scipy(
            trajectories=trajectories,
            observable_idx=observable_idx,
        )

        workbook_sheets[observable_name] = pd.DataFrame(
            matrix,
            index=file_names,
            columns=file_names,
        )

    print(f"\n[{folder}] writing workbook: {output_path}")
    write_result_workbook(output_path, workbook_sheets)
    print(f"[{folder}] done")
    return output_path


scipy_result_paths = [process_folder_scipy(folder) for folder in FOLDERS]
scipy_result_paths



=== paths1: SciPy pairwise W1 with burn-in=50,000 ===
[paths1] loaded 100/1000 files
[paths1] loaded 200/1000 files
[paths1] loaded 300/1000 files
[paths1] loaded 400/1000 files
[paths1] loaded 500/1000 files
[paths1] loaded 600/1000 files
[paths1] loaded 700/1000 files
[paths1] loaded 800/1000 files
[paths1] loaded 900/1000 files
[paths1] loaded 1000/1000 files

[paths1] Observable: theta1
    pairwise rows 10/1000
    pairwise rows 20/1000
    pairwise rows 30/1000
    pairwise rows 40/1000
    pairwise rows 50/1000
    pairwise rows 60/1000
    pairwise rows 70/1000
    pairwise rows 80/1000
    pairwise rows 90/1000
    pairwise rows 100/1000
    pairwise rows 110/1000
    pairwise rows 120/1000
    pairwise rows 130/1000
    pairwise rows 140/1000
    pairwise rows 150/1000
    pairwise rows 160/1000
    pairwise rows 170/1000
    pairwise rows 180/1000
    pairwise rows 190/1000
    pairwise rows 200/1000
    pairwise rows 210/1000
    pairwise rows 220/1000
    pairwise rows 23

KeyboardInterrupt: 